# Test de pipeline.py
Lancer avec le venv du projet (`../.venv-tfm`, après `uv pip install openml`). Un seul modèle par kernel sur Mac (XGBoost **ou** TabICL).

In [1]:
import pandas as pd
from chargement import liste_tabarena, charge
from pipeline import pipeline
from xgboost import XGBClassifier

modele = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                       n_jobs=1, random_state=0, tree_method="hist")

In [2]:
# 1. Les 38 datasets de classification de TabArena
TABARENA = liste_tabarena()
TABARENA

,name,tid,NumberOfInstances,NumberOfFeatures,NumberOfClasses
0,blood-transfusion-service-center,363621,748,5,2
1,diabetes,363629,768,9,2
2,anneal,363614,898,39,5
3,credit-g,363626,1000,21,2
4,maternal_health_risk,363685,1014,7,3
5,qsar-biodeg,363696,1054,42,2
6,website_phishing,363707,1353,10,3
7,Fitness_Club,363671,1500,7,2
8,MIC,363711,1699,112,8
9,Is-this-a-good-customer,363682,1723,14,2


In [3]:
# 2. Fitness_Club depuis OpenML (le doublon est supprimé -> 1499 lignes, comme votre CSV)
X, y = charge("Fitness_Club", TABARENA)
res = pd.concat([pipeline("Fitness_Club", X, y, modele, m, score="lac") for m in ["split", "cv+"]])
res

,dataset,modele,methode,score,alpha,graine,lam,k_reg,coverage,avg_set_size,...,empty_rate,accuracy,precision_macro,recall_macro,f1_macro,n_samples,n_doublons,n_features,n_classes,temps_s
0,Fitness_Club,XGBClassifier,split,lac,0.05,90,None,None,0.949333,1.570667,...,0.0,0.762667,0.721145,0.686227,0.697550,1499,1,6,2,0.012414
1,Fitness_Club,XGBClassifier,split,lac,0.10,90,None,None,0.904000,1.384000,...,0.0,0.762667,0.721145,0.686227,0.697550,1499,1,6,2,0.012414
2,Fitness_Club,XGBClassifier,split,lac,0.20,90,None,None,0.805333,1.114667,...,0.0,0.762667,0.721145,0.686227,0.697550,1499,1,6,2,0.012414
0,Fitness_Club,XGBClassifier,cv+,lac,0.05,90,None,None,0.970667,1.594667,...,0.0,0.765333,0.723977,0.693083,0.703728,1499,1,6,2,0.132585
1,Fitness_Club,XGBClassifier,cv+,lac,0.10,90,None,None,0.925333,1.408000,...,0.0,0.765333,0.723977,0.693083,0.703728,1499,1,6,2,0.132585
2,Fitness_Club,XGBClassifier,cv+,lac,0.20,90,None,None,0.813333,1.098667,...,0.0,0.765333,0.723977,0.693083,0.703728,1499,1,6,2,0.132585


In [4]:
# 3. Anciens résultats, pour comparer (doivent être identiques)
old = pd.read_csv("../Projet_RI_TabFn/results_xgboost.csv")
old[(old.source == "principal") & (old.methode != "Jackknife+")]

,modele,methode,alpha,coverage,avg_set_size,singleton_rate,empty_rate,accuracy,precision_macro,recall_macro,f1_macro,temps_s,graine,source
0,XGBoost,Split,0.05,0.949333,1.570667,0.429333,0.0,0.762667,0.721145,0.686227,0.697550,0.011499,90,principal
1,XGBoost,CV+,0.05,0.970667,1.594667,0.405333,0.0,0.765333,0.723977,0.693083,0.703728,0.118700,90,principal
3,XGBoost,Split,0.10,0.904000,1.384000,0.616000,0.0,0.762667,0.721145,0.686227,0.697550,0.011499,90,principal
4,XGBoost,CV+,0.10,0.925333,1.408000,0.592000,0.0,0.765333,0.723977,0.693083,0.703728,0.118700,90,principal
6,XGBoost,Split,0.20,0.805333,1.114667,0.885333,0.0,0.762667,0.721145,0.686227,0.697550,0.011499,90,principal
7,XGBoost,CV+,0.20,0.813333,1.098667,0.901333,0.0,0.765333,0.723977,0.693083,0.703728,0.118700,90,principal


In [5]:
# 4. Les 3 scores x 3 alphas sur Fitness_Club
pipeline("Fitness_Club", X, y, modele, "split")

,dataset,modele,methode,score,alpha,graine,lam,k_reg,coverage,avg_set_size,...,empty_rate,accuracy,precision_macro,recall_macro,f1_macro,n_samples,n_doublons,n_features,n_classes,temps_s
0,Fitness_Club,XGBClassifier,split,lac,0.05,90,NaN,NaN,0.949333,1.570667,...,0.0,0.762667,0.721145,0.686227,0.69755,1499,1,6,2,0.011067
1,Fitness_Club,XGBClassifier,split,lac,0.10,90,NaN,NaN,0.904000,1.384000,...,0.0,0.762667,0.721145,0.686227,0.69755,1499,1,6,2,0.011067
2,Fitness_Club,XGBClassifier,split,lac,0.20,90,NaN,NaN,0.805333,1.114667,...,0.0,0.762667,0.721145,0.686227,0.69755,1499,1,6,2,0.011067
3,Fitness_Club,XGBClassifier,split,aps,0.05,90,NaN,NaN,0.952000,1.829333,...,0.0,0.762667,0.721145,0.686227,0.69755,1499,1,6,2,0.011067
4,Fitness_Club,XGBClassifier,split,aps,0.10,90,NaN,NaN,0.933333,1.698667,...,0.0,0.762667,0.721145,0.686227,0.69755,1499,1,6,2,0.011067
5,Fitness_Club,XGBClassifier,split,aps,0.20,90,NaN,NaN,0.789333,1.216000,...,0.0,0.762667,0.721145,0.686227,0.69755,1499,1,6,2,0.011067
6,Fitness_Club,XGBClassifier,split,raps,0.05,90,0.01,1.0,0.952000,1.829333,...,0.0,0.762667,0.721145,0.686227,0.69755,1499,1,6,2,0.011067
7,Fitness_Club,XGBClassifier,split,raps,0.10,90,0.01,1.0,0.933333,1.698667,...,0.0,0.762667,0.721145,0.686227,0.69755,1499,1,6,2,0.011067
8,Fitness_Club,XGBClassifier,split,raps,0.20,90,0.01,1.0,0.789333,1.216000,...,0.0,0.762667,0.721145,0.686227,0.69755,1499,1,6,2,0.011067


In [6]:
# 5. Un dataset multi-classe
X, y = charge("anneal", TABARENA)
pipeline("anneal", X, y, modele, "cv+")

,dataset,modele,methode,score,alpha,graine,lam,k_reg,coverage,avg_set_size,...,empty_rate,accuracy,precision_macro,recall_macro,f1_macro,n_samples,n_doublons,n_features,n_classes,temps_s
0,anneal,XGBClassifier,cv+,lac,0.05,90,NaN,NaN,0.959459,0.959459,...,0.040541,0.995495,0.992,0.998817,0.995325,886,12,38,5,0.443553
1,anneal,XGBClassifier,cv+,lac,0.10,90,NaN,NaN,0.900901,0.900901,...,0.099099,0.995495,0.992,0.998817,0.995325,886,12,38,5,0.443553
2,anneal,XGBClassifier,cv+,lac,0.20,90,NaN,NaN,0.833333,0.833333,...,0.166667,0.995495,0.992,0.998817,0.995325,886,12,38,5,0.443553
3,anneal,XGBClassifier,cv+,aps,0.05,90,NaN,NaN,0.990991,2.752252,...,0.009009,0.995495,0.992,0.998817,0.995325,886,12,38,5,0.443553
4,anneal,XGBClassifier,cv+,aps,0.10,90,NaN,NaN,0.945946,2.500000,...,0.054054,0.995495,0.992,0.998817,0.995325,886,12,38,5,0.443553
5,anneal,XGBClassifier,cv+,aps,0.20,90,NaN,NaN,0.806306,2.099099,...,0.193694,0.995495,0.992,0.998817,0.995325,886,12,38,5,0.443553
6,anneal,XGBClassifier,cv+,raps,0.05,90,0.01,1.0,0.990991,1.103604,...,0.009009,0.995495,0.992,0.998817,0.995325,886,12,38,5,0.443553
7,anneal,XGBClassifier,cv+,raps,0.10,90,0.01,1.0,0.950450,1.040541,...,0.049550,0.995495,0.992,0.998817,0.995325,886,12,38,5,0.443553
8,anneal,XGBClassifier,cv+,raps,0.20,90,0.01,1.0,0.806306,0.869369,...,0.193694,0.995495,0.992,0.998817,0.995325,886,12,38,5,0.443553


## 6. Tous les jeux de TabArena (split et cv+)
Chaque résultat est ajouté à `results_pipeline.csv` dès qu'il est calculé. Les combinaisons (dataset, modèle, méthode) déjà présentes dans le CSV sont sautées : on peut interrompre et relancer. Jackknife+ est lancé dans une autre cellule. Pour tester : `JEUX = TABARENA["name"][:2]`.

In [7]:
import os, time

JEUX = TABARENA["name"]             # [:2] pour tester
modeles = [modele]
METHODES = ["split", "cv+"]
FICHIER = "results_pipeline.csv"

deja = set()
if os.path.exists(FICHIER):
    deja = set(map(tuple, pd.read_csv(FICHIER)[["dataset", "modele", "methode"]].drop_duplicates().values))

echecs = []
for nom in JEUX:
    a_faire = [(m, meth) for m in modeles for meth in METHODES if (nom, type(m).__name__, meth) not in deja]
    if not a_faire:
        print(f"{nom} : déjà fait")
        continue
    try:
        X, y = charge(nom, TABARENA)
    except Exception as e:
        print(f"ÉCHEC chargement {nom} : {e}")
        echecs += [(nom, type(m).__name__, meth, f"chargement : {e}") for m, meth in a_faire]
        continue
    for m, meth in a_faire:
        t0 = time.time()
        try:
            df = pipeline(nom, X, y, m, meth)
            df.to_csv(FICHIER, mode="a", header=not os.path.exists(FICHIER), index=False)
            print(f"{nom} / {type(m).__name__} / {meth} : {time.time() - t0:.1f} s")
        except Exception as e:
            print(f"ÉCHEC {nom} / {type(m).__name__} / {meth} ({time.time() - t0:.1f} s) : {e}")
            echecs.append((nom, type(m).__name__, meth, str(e)))

print(f"{len(echecs)} échec(s)")
echecs

blood-transfusion-service-center : déjà fait
diabetes : déjà fait
anneal / XGBClassifier / split : 0.1 s
anneal / XGBClassifier / cv+ : 0.5 s
credit-g / XGBClassifier / split : 0.0 s
credit-g / XGBClassifier / cv+ : 0.2 s
maternal_health_risk / XGBClassifier / split : 0.0 s
maternal_health_risk / XGBClassifier / cv+ : 0.2 s
qsar-biodeg / XGBClassifier / split : 0.1 s
qsar-biodeg / XGBClassifier / cv+ : 0.5 s
website_phishing / XGBClassifier / split : 0.0 s
website_phishing / XGBClassifier / cv+ : 0.2 s
Fitness_Club / XGBClassifier / split : 0.0 s
Fitness_Club / XGBClassifier / cv+ : 0.1 s
MIC / XGBClassifier / split : 0.5 s
MIC / XGBClassifier / cv+ : 4.8 s
Is-this-a-good-customer / XGBClassifier / split : 0.1 s
Is-this-a-good-customer / XGBClassifier / cv+ : 0.2 s
Marketing_Campaign / XGBClassifier / split : 0.1 s
Marketing_Campaign / XGBClassifier / cv+ : 0.4 s
hazelnut-spread-contaminant-detection / XGBClassifier / split : 0.2 s
hazelnut-spread-contaminant-detection / XGBClassifier 

[]